# 실습 4. OpenAI API 챗봇 서비스 — 멀티턴 메모리 (Day 2 / 150분)

> 시나리오: **대화 히스토리를 누적하는 멀티턴 챗봇** (페르소나 자유)
>
> `openai` 패키지의 messages 리스트가 곧 *대화 메모리*다.
> (LangChain 의 ConversationBufferMemory 와 같은 개념을 직접 구현한다.)

## Task
1. 히스토리 챗봇 — 대화 누적 구조 (`/reset` 초기화)
2. 페르소나 변경 (system prompt) · temperature 0 vs 0.7
3. 환각 발견 — 모르는 사실 묻기 → system 으로 줄이기
4. 토큰/비용 모니터링 — 100턴 누적 비용 예측 + trim 전략

## 0) 준비

In [38]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."

## 1) Task 1 — 히스토리 챗봇 (대화 누적)

`history` 리스트가 메모리다. 매 호출에 system + 전체 history 를 함께 보낸다.
`chat()` 을 여러 번 호출하면 앞선 대화를 기억한다. `reset()` 으로 초기화.

In [39]:
history = []          # [{"role": "user"/"assistant", "content": ...}]
usage_log = []        # 턴별 토큰 기록 (Task 4)

def reset():
    history.clear()
    usage_log.clear()
    print("(대화 초기화)")

def chat(message: str, temperature: float = 0.3) -> str:
    history.append({"role": "user", "content": message})
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM}, *history],
    )
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    usage_log.append(resp.usage.total_tokens)
    return answer

In [40]:
reset()
print(chat("니 이름은 홍길동이야. 기억해줘."))
print(chat("방금 내가 말한 니 이름이 뭐였지?"))   # 앞 턴을 기억하는지 확인

(대화 초기화)
확인 필요.
홍길동이었습니다.


In [41]:
# 대화입력 위젯: 입력창, 전송 버튼, 종료 버튼을 제공합니다.
from IPython.display import display
import ipywidgets as widgets

output = widgets.Output(layout={'border': '1px solid #ddd'})
input_box = widgets.Text(placeholder='메시지를 입력하세요. (/reset /history /exit)', description='You:')
send_btn = widgets.Button(description='Send', button_style='primary')
exit_btn = widgets.Button(description='Exit', button_style='danger')

def handle_message(msg: str):
    msg = msg.strip()
    if not msg:
        return
    if msg.lower() in ('/exit', '/quit'):
        with output:
            print('세션 종료')
        input_box.disabled = True
        send_btn.disabled = True
        exit_btn.disabled = True
        return
    if msg.lower() in ('/reset', '/r'):
        reset()
        with output:
            print('(대화 초기화)')
        return
    if msg.lower() in ('/history', '/h'):
        from pprint import pprint
        with output:
            pprint(history)
        return
    try:
        reply = chat(msg)
    except Exception as e:
        with output:
            print('오류 발생:', e)
        return
    with output:
        print('You:', msg)
        print('Assistant:', reply)

def on_send(b):
    handle_message(input_box.value)
    input_box.value = ''

def on_submit(sender):
    handle_message(sender.value)
    sender.value = ''

send_btn.on_click(on_send)
exit_btn.on_click(lambda b: handle_message('/exit'))
input_box.on_submit(on_submit)

controls = widgets.HBox([input_box, send_btn, exit_btn])
display(widgets.VBox([controls, output]))

/tmp/ipykernel_51099/3098988493.py:51: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_submit)


## 2) Task 2 — 페르소나 변경 · temperature 비교

`SYSTEM` 만 바꿔 말투가 바뀐다. temperature 0(일관) vs 0.7(다양).

In [42]:
SYSTEM = "당신은 매우 엄격하고 격식 있는 사서입니다. 짧고 단정하게 답합니다."
reset()
print("[엄격한 사서]", chat("주말에 뭐하면 좋을까?", temperature=0))

SYSTEM = "당신은 친근한 동네 친구입니다. 반말로 편하게 답합니다."
reset()
print("[친근한 친구]", chat("주말에 뭐하면 좋을까?", temperature=0.7))
# TODO: 같은 질문을 temperature 0 / 0.7 로 3회씩 — 답변 다양성 차이를 메모

(대화 초기화)


[엄격한 사서] 독서, 산책, 영화 관람, 취미 활동 등을 추천합니다.
(대화 초기화)
[친근한 친구] 주말에 뭔가 즐거운 거 찾고 있구나! 날씨 좋으면 공원이나 카페 가서 산책하면서 책 읽는 것도 좋고, 친구들이랑 영화 보거나 보드게임 하는 것도 재밌어. 요즘 인기 있는 맛집 탐방하는 것도 추천해! 너는 어떤 걸 좋아해?


## 3) Task 3 — 환각 발견 → system 으로 줄이기

존재하지 않는 사실을 물어 *지어내는지* 본다. 그 뒤 system 을 강화해 5회 반복.

In [43]:
SYSTEM = "당신은 창의적인 AI 어시스턴트입니다."
reset()
print("[가드 없음]", chat("외계인은 어떻게 생겼어?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 절대 지어내지 말고 '확인 필요'라고만 답하세요."
    #"혹은 확인되지 않은 정보는 가능한 방법 명시하세요"
)
reset()
print("[가드 강화]", chat("외계인은 어떻게 생겼어?"))
# TODO: 환각 발견 로그 + system 수정 히스토리를 표로 남긴다 (5회 반복)


(대화 초기화)
[가드 없음] 외계인의 모습은 과학적 증거가 없기 때문에 여러 가지 상상력에 기반한 이론과 묘사가 존재합니다. 영화, 책, 만화 등에서 다양한 형태로 그려지는데, 일반적으로는 다음과 같은 특징을 가집니다:

1. **인간형**: 두 팔, 두 다리, 머리와 몸통이 있는 형태. 이들은 종종 큰 눈과 작은 코, 입을 가지고 있는 모습으로 묘사됩니다.

2. **비인간형**: 곤충처럼 생기거나, 물고기, 파충류와 같은 형태로 그려지기도 합니다. 이들은 종종 비정상적으로 많은 다리나 더듬이를 가지고 있습니다.

3. **에너지 형태**: 어떤 이론에서는 외계인이 물리적인 형태가 아닌 에너지나 빛의 형태로 존재할 수 있다고 주장하기도 합니다.

4. **다양한 색상과 질감**: 외계인의 피부 색상이나 질감은 매우 다양하게 상상될 수 있으며, 금속성, 비늘, 혹은 투명한 형태로 묘사되기도 합니다.

결국 외계인의 모습은 과학적 사실보다는 상상력과 문화적 배경에 따라 다르게 표현됩니다. 실제로 외계 생명체가 존재한다면 그 모습은 우리가 상상하는 것과는 전혀 다를 수 있습니다.
(대화 초기화)
[가드 강화] 확인 필요.


## 4) Task 4 — 토큰/비용 모니터링 + trim 전략

In [46]:
PRICE = 0.30 / 1_000_000   # 입출력 혼합 평균 가정 단가 ($)

total = sum(usage_log)
print("턴별 토큰:", usage_log)
print(f"누적 토큰: {total}, 누적 비용: ${total * PRICE:.5f}")

# 히스토리는 누적될수록 매 호출 토큰이 선형 증가 → 100턴이면 비용 급증
if usage_log:
    per_turn = total / len(usage_log)
    print(f"100턴 단순 추정: ${per_turn * 100 * PRICE:.4f} (실제로는 더 큼 — 누적 때문)")

def trim(history, keep=6):
    """최근 keep 개 메시지만 유지 — 메모리 trim 전략."""
    return history[-keep:]

def estimate_tokens_from_messages(messages, model=MODEL):
    """메시지 리스트의 토큰 수를 대략 추정한다. tiktoken 이 있으면 더 정확하다."""
    try:
        import tiktoken
        enc = tiktoken.encoding_for_model(model)
        count_text = lambda text: len(enc.encode(text))
        per_message_overhead = 4
        final_overhead = 2
    except Exception:
        count_text = lambda text: max(1, len(text) // 4)
        per_message_overhead = 0
        final_overhead = 0

    total_tokens = final_overhead
    for msg in messages:
        total_tokens += per_message_overhead + count_text(msg.get("content", ""))
        if msg.get("name"):
            total_tokens -= 1
    return total_tokens

def summarize_history(messages):
    """오래된 대화를 짧은 메모로 압축하는 summary memory 예시."""
    if not messages:
        return "대화 요약 없음"

    user_topics = [m["content"] for m in messages if m["role"] == "user"]
    assistant_points = [m["content"] for m in messages if m["role"] == "assistant"]
    parts = []

    if user_topics:
        first = user_topics[0]
        parts.append(f"시작 요청: {first[:60]}{'...' if len(first) > 60 else ''}")
    if len(user_topics) > 1:
        parts.append(f"추가 요청 수: {len(user_topics) - 1}")
    if assistant_points:
        last = assistant_points[-1]
        parts.append(f"최근 응답 포인트: {last[:60]}{'...' if len(last) > 60 else ''}")

    return " | ".join(parts) if parts else "대화 요약 없음"

full_history = history[:]
trimmed_history = trim(full_history, keep=4)
older_history = full_history[:-4]
summary_text = summarize_history(older_history)
summary_history = ([{"role": "system", "content": f"이전 대화 요약: {summary_text}"}] + trimmed_history)

full_tokens = estimate_tokens_from_messages(full_history)
trimmed_tokens = estimate_tokens_from_messages(trimmed_history)
summary_tokens = estimate_tokens_from_messages(summary_history)

print(f"전체 history 메시지 수: {len(full_history)} / 추정 토큰: {full_tokens}")
print(f"trim(4) 후 메시지 수: {len(trimmed_history)} / 추정 토큰: {trimmed_tokens}")
print(f"summary+trim 메시지 수: {len(summary_history)} / 추정 토큰: {summary_tokens}")


if summary_tokens >= full_tokens:
    print("주의: 현재 대화는 짧아서 summary 오버헤드가 절감분보다 큽니다. 긴 대화일수록 summary 가 유리합니다.")
if full_tokens:
    print(f"trim 절감률: {(1 - trimmed_tokens / full_tokens) * 100:.1f}%")
    print(f"summary+trim 절감률: {(1 - summary_tokens / full_tokens) * 100:.1f}%")

print("\n[summary memory 예시]")
print(summary_history[0]["content"])


턴별 토큰: [64]
누적 토큰: 64, 누적 비용: $0.00002
100턴 단순 추정: $0.0019 (실제로는 더 큼 — 누적 때문)
전체 history 메시지 수: 2 / 추정 토큰: 22
trim(4) 후 메시지 수: 2 / 추정 토큰: 22
summary+trim 메시지 수: 3 / 추정 토큰: 38
주의: 현재 대화는 짧아서 summary 오버헤드가 절감분보다 큽니다. 긴 대화일수록 summary 가 유리합니다.
trim 절감률: 0.0%
summary+trim 절감률: -72.7%

[summary memory 예시]
이전 대화 요약: 대화 요약 없음


### Summary memory trade-off

- 장점: 오래된 대화의 핵심만 압축해 유지하므로, trim만 쓸 때보다 맥락 유실이 적다.
- 단점: 요약을 만드는 순간 토큰이 추가로 들고, 원문 디테일은 사라질 수 있다.
- 주의: 대화가 짧을 때는 summary 오버헤드가 절감 효과보다 커질 수 있다. 긴 대화에서 더 유리하다.
- 권장: 최근 턴은 `trim`으로 유지하고, 오래된 사실·선호는 `summary`에 누적하는 하이브리드 방식.


## 회고 / 산출물
- [ ] 환각 발견 로그 + 수정 히스토리
- [ ] 페르소나별 답변 비교 메모
- [x] 100턴 누적 비용 예측 + trim/summary 전략 한 줄
